# Nettoyage des données — Credit Score Classification

Ce notebook nettoie le jeu de données bancaire Kaggle **Credit Score Classification**
(`data/train.csv`, 100 000 lignes, 28 colonnes) afin de produire une version
propre et exploitable pour la modélisation : `data/train_cleaned.csv`.

**Étapes réalisées :**
1. Chargement et diagnostic initial
2. Suppression des valeurs placeholders
3. Nettoyage des formats numériques
4. Correction des valeurs aberrantes
5. Imputation des valeurs manquantes
6. Traitement de `Type_of_Loan`
7. Conversion de `Credit_History_Age` en mois
8. Imputation des colonnes restantes (médiane / mode par client)
9. Vérification finale et résumé des corrections
10. Export du jeu de données nettoyé


In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)


## 1. Chargement et diagnostic initial

On charge le fichier brut `data/train.csv` et on établit un premier diagnostic :
dimensions, types de données, valeurs manquantes et aperçu des premières lignes.


In [2]:
df = pd.read_csv('../data/train.csv', low_memory=False)
print(f"Dimensions du dataset : {df.shape[0]} lignes x {df.shape[1]} colonnes")


Dimensions du dataset : 100000 lignes x 28 colonnes


In [3]:
df.dtypes


ID                              str
Customer_ID                     str
Month                           str
Name                            str
Age                             str
SSN                             str
Occupation                      str
Annual_Income                   str
Monthly_Inhand_Salary       float64
Num_Bank_Accounts             int64
Num_Credit_Card               int64
Interest_Rate                 int64
Num_of_Loan                     str
Type_of_Loan                    str
Delay_from_due_date           int64
Num_of_Delayed_Payment          str
Changed_Credit_Limit            str
Num_Credit_Inquiries        float64
Credit_Mix                      str
Outstanding_Debt                str
Credit_Utilization_Ratio    float64
Credit_History_Age              str
Payment_of_Min_Amount           str
Total_EMI_per_month         float64
Amount_invested_monthly         str
Payment_Behaviour               str
Monthly_Balance                 str
Credit_Score                

In [4]:
# Valeurs manquantes AVANT nettoyage (déclarées comme NaN dans le CSV brut)
missing_avant = df.isnull().sum()
missing_avant = missing_avant[missing_avant > 0].sort_values(ascending=False)
print("Colonnes avec des valeurs manquantes explicites (NaN) :")
missing_avant


Colonnes avec des valeurs manquantes explicites (NaN) :


Monthly_Inhand_Salary      15002
Type_of_Loan               11408
Name                        9985
Credit_History_Age          9030
Num_of_Delayed_Payment      7002
Amount_invested_monthly     4479
Num_Credit_Inquiries        1965
Monthly_Balance             1200
dtype: int64

In [5]:
df.head(10)


,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3,7,11.27,4.0,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",-1,NaN,11.27,4.0,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3,7,_,4.0,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",5,4,6.27,4.0,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",6,NaN,11.27,4.0,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good
5,0x1607,CUS_0xd40,June,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",8,4,9.27,4.0,Good,809.98,27.262259,22 Years and 6 Months,No,49.574949,62.430172331195294,!@9#%8,340.4792117872438,Good
6,0x1608,CUS_0xd40,July,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3,8_,11.27,4.0,Good,809.98,22.537593,22 Years and 7 Months,No,49.574949,178.3440674122349,Low_spent_Small_value_payments,244.5653167062043,Good
7,0x1609,CUS_0xd40,August,NaN,23,#F%$D@*&8,Scientist,19114.12,1824.843333,3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3,6,11.27,4.0,Good,809.98,23.933795,NaN,No,49.574949,24.785216509052056,High_spent_Medium_value_payments,358.12416760938714,Standard
8,0x160e,CUS_0x21b1,January,Rick Rothackerj,28_,004-07-5839,_______,34847.84,3037.986667,2,4,6,1,Credit-Builder Loan,3,4,5.42,2.0,Good,605.03,24.464031,26 Years and 7 Months,No,18.816215,104.291825168246,Low_spent_Small_value_payments,470.69062692529184,Standard
9,0x160f,CUS_0x21b1,February,Rick Rothackerj,28,004-07-5839,Teacher,34847.84,3037.986667,2,4,6,1,Credit-Builder Loan,7,1,7.42,2.0,Good,605.03,38.550848,26 Years and 8 Months,No,18.816215,40.39123782853101,High_spent_Large_value_payments,484.5912142650067,Good


In [6]:
# On garde une copie du nombre total de cellules manquantes pour le résumé final
n_cells_total = df.shape[0] * df.shape[1]
missing_total_initial = int(df.isnull().sum().sum())
print(f"Cellules manquantes (NaN) au chargement : {missing_total_initial} / {n_cells_total}")


Cellules manquantes (NaN) au chargement : 60071 / 2800000


## 2. Suppression des valeurs placeholders

Le dataset contient des codes utilisés à la place de vraies valeurs manquantes :
`_`, `NM`, `!@9#%8`, `#F%$D@*&8`, `__10000__`, `_______`, `NA`.

On les remplace toutes par `NaN` afin qu'elles soient traitées comme de véritables
valeurs manquantes dans les étapes suivantes.


In [7]:
PLACEHOLDERS = ['_', 'NM', '!@9#%8', '#F%$D@*&8', '__10000__', '_______', 'NA']

# Nombre de placeholders détectés par colonne, avant remplacement
placeholder_counts = {}
for col in df.columns:
    if df[col].dtype == object:
        cnt = df[col].isin(PLACEHOLDERS).sum()
        if cnt > 0:
            placeholder_counts[col] = cnt

pd.Series(placeholder_counts, name='nb_placeholders').sort_values(ascending=False)


Series([], Name: nb_placeholders, dtype: object)

In [8]:
df.replace(PLACEHOLDERS, np.nan, inplace=True)

missing_apres_placeholders = int(df.isnull().sum().sum())
print(f"Cellules manquantes avant remplacement des placeholders : {missing_total_initial}")
print(f"Cellules manquantes après remplacement des placeholders : {missing_apres_placeholders}")
print(f"Nouvelles valeurs manquantes créées : {missing_apres_placeholders - missing_total_initial}")


Cellules manquantes avant remplacement des placeholders : 60071
Cellules manquantes après remplacement des placeholders : 118903
Nouvelles valeurs manquantes créées : 58832


## 3. Nettoyage des formats numériques

Plusieurs colonnes numériques sont stockées en texte et contiennent un underscore
parasite en fin (ou en début) de valeur, par exemple `"23_"` au lieu de `23`.
On retire ces underscores puis on convertit les colonnes en `float` :

`Age`, `Annual_Income`, `Num_of_Loan`, `Num_of_Delayed_Payment`, `Outstanding_Debt`,
`Changed_Credit_Limit`, `Amount_invested_monthly`, `Monthly_Balance`.


In [9]:
NUMERIC_COLS_TO_CLEAN = [
    'Age',
    'Annual_Income',
    'Num_of_Loan',
    'Num_of_Delayed_Payment',
    'Outstanding_Debt',
    'Changed_Credit_Limit',
    'Amount_invested_monthly',
    'Monthly_Balance',
]

dtypes_avant = df[NUMERIC_COLS_TO_CLEAN].dtypes.copy()

def nettoyer_numerique(serie):
    """Retire les underscores parasites et convertit en float."""
    return pd.to_numeric(
        serie.astype(str).str.replace('_', '', regex=False).str.strip(),
        errors='coerce'
    )

for col in NUMERIC_COLS_TO_CLEAN:
    df[col] = nettoyer_numerique(df[col])

dtypes_apres = df[NUMERIC_COLS_TO_CLEAN].dtypes.copy()

pd.DataFrame({'dtype_avant': dtypes_avant, 'dtype_apres': dtypes_apres})


,dtype_avant,dtype_apres
Age,str,int64
Annual_Income,str,float64
Num_of_Loan,str,int64
Num_of_Delayed_Payment,str,float64
Outstanding_Debt,str,float64
Changed_Credit_Limit,str,float64
Amount_invested_monthly,str,float64
Monthly_Balance,str,float64


In [10]:
df[NUMERIC_COLS_TO_CLEAN].describe()


,Age,Annual_Income,Num_of_Loan,Num_of_Delayed_Payment,Outstanding_Debt,Changed_Credit_Limit,Amount_invested_monthly,Monthly_Balance
count,100000.000000,1.000000e+05,100000.000000,92998.000000,100000.000000,97909.000000,91216.000000,9.880000e+04
mean,110.649700,1.764157e+05,3.009960,30.923342,1426.220376,10.389025,195.539456,-3.036437e+22
std,686.244717,1.429618e+06,62.647879,226.031892,1155.129026,6.789496,199.564527,3.181295e+24
min,-500.000000,7.005930e+03,-100.000000,-3.000000,0.230000,-6.490000,0.000000,-3.333333e+26
25%,24.000000,1.945750e+04,1.000000,9.000000,566.072500,5.320000,72.236692,2.700922e+02
50%,33.000000,3.757861e+04,3.000000,14.000000,1166.155000,9.400000,128.954538,3.367192e+02
75%,42.000000,7.279092e+04,5.000000,18.000000,1945.962500,14.870000,236.815814,4.702202e+02
max,8698.000000,2.419806e+07,1496.000000,4397.000000,4998.070000,36.970000,1977.326102,1.602041e+03


## 4. Correction des valeurs aberrantes

Certaines colonnes numériques contiennent des valeurs hors de plages réalistes
(âges négatifs, taux d'intérêt absurdes, etc.). Pour chaque colonne, les valeurs
hors plage sont d'abord mises à `NaN`, puis remplacées par la **médiane du même
`Customer_ID`** (chaque client apparaît sur plusieurs mois, donc sa médiane
personnelle est une estimation fiable). S'il ne reste aucune valeur valide pour
un client donné, la médiane globale de la colonne est utilisée en secours.

Plages considérées comme valides :

| Colonne | Plage valide |
|---|---|
| Age | 14 - 100 |
| Interest_Rate | 0 - 40 |
| Num_Bank_Accounts | 0 - 15 |
| Num_Credit_Card | 0 - 15 |
| Num_of_Loan | 0 - 15 |
| Annual_Income | 1000 - 300000 |
| Num_Credit_Inquiries | 0 - 20 |


In [11]:
VALID_RANGES = {
    'Age': (14, 100),
    'Interest_Rate': (0, 40),
    'Num_Bank_Accounts': (0, 15),
    'Num_Credit_Card': (0, 15),
    'Num_of_Loan': (0, 15),
    'Annual_Income': (1000, 300000),
    'Num_Credit_Inquiries': (0, 20),
}

# Nombre de valeurs hors plage AVANT correction
outliers_avant = {}
for col, (lo, hi) in VALID_RANGES.items():
    hors_plage = ~df[col].between(lo, hi) & df[col].notna()
    outliers_avant[col] = int(hors_plage.sum())

pd.Series(outliers_avant, name='nb_valeurs_hors_plage')


Age                     2776
Interest_Rate           2034
Num_Bank_Accounts       1336
Num_Credit_Card         2268
Num_of_Loan             4348
Annual_Income            993
Num_Credit_Inquiries    1650
Name: nb_valeurs_hors_plage, dtype: int64

In [12]:
def corriger_aberrantes(dataframe, colonne, borne_min, borne_max):
    """Remplace les valeurs hors [borne_min, borne_max] par la médiane du même
    Customer_ID (ou la médiane globale si le client n'a aucune valeur valide)."""
    serie = dataframe[colonne].copy()
    hors_plage = ~serie.between(borne_min, borne_max) & serie.notna()
    serie[hors_plage] = np.nan

    mediane_globale = serie.median()

    # Médiane calculée uniquement sur les valeurs valides de chaque client
    valeurs_valides = serie.where(serie.between(borne_min, borne_max))
    mediane_client = valeurs_valides.groupby(dataframe['Customer_ID']).transform('median')

    serie = serie.fillna(mediane_client)
    serie = serie.fillna(mediane_globale)
    return serie

for col, (lo, hi) in VALID_RANGES.items():
    df[col] = corriger_aberrantes(df, col, lo, hi)


In [13]:
# Vérification APRES correction : il ne doit plus rester de valeurs hors plage
outliers_apres = {}
for col, (lo, hi) in VALID_RANGES.items():
    hors_plage = ~df[col].between(lo, hi) & df[col].notna()
    outliers_apres[col] = int(hors_plage.sum())

comparaison_outliers = pd.DataFrame({
    'avant': outliers_avant,
    'apres': outliers_apres,
})
comparaison_outliers


,avant,apres
Age,2776,0
Interest_Rate,2034,0
Num_Bank_Accounts,1336,0
Num_Credit_Card,2268,0
Num_of_Loan,4348,0
Annual_Income,993,0
Num_Credit_Inquiries,1650,0


In [14]:
df[list(VALID_RANGES.keys())].describe()


,Age,Interest_Rate,Num_Bank_Accounts,Num_Credit_Card,Num_of_Loan,Annual_Income,Num_Credit_Inquiries
count,100000.000000,100000.00000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,33.314600,14.53208,5.368840,5.533910,3.532880,50518.846732,5.779590
std,10.773909,8.74133,2.593273,2.067712,2.446356,38335.621871,3.861453
min,14.000000,1.00000,0.000000,0.000000,0.000000,7005.930000,0.000000
25%,24.000000,7.00000,3.000000,4.000000,2.000000,19344.270000,3.000000
50%,33.000000,13.00000,5.000000,5.000000,3.000000,37002.580000,5.000000
75%,42.000000,20.00000,7.000000,7.000000,5.000000,71689.680000,8.000000
max,100.000000,34.00000,11.000000,15.000000,9.000000,277803.000000,17.000000


## 5. Imputation des valeurs manquantes

Chaque `Customer_ID` correspond à un même client suivi sur plusieurs mois. Les
caractéristiques suivantes sont normalement stables ou connues pour ce client sur
d'autres mois : `Monthly_Inhand_Salary`, `Occupation`, `Credit_Mix`,
`Type_of_Loan`, `Payment_Behaviour`.

On applique une propagation avant (`ffill`) puis arrière (`bfill`), groupée par
`Customer_ID`, pour combler les valeurs manquantes à partir des autres mois du
même client.


In [15]:
COLS_A_IMPUTER = [
    'Monthly_Inhand_Salary',
    'Occupation',
    'Credit_Mix',
    'Type_of_Loan',
    'Payment_Behaviour',
]

missing_avant_imputation = df[COLS_A_IMPUTER].isnull().sum()
missing_avant_imputation


Monthly_Inhand_Salary    15002
Occupation                7062
Credit_Mix               20195
Type_of_Loan             11408
Payment_Behaviour         7600
dtype: int64

In [16]:
df[COLS_A_IMPUTER] = df.groupby('Customer_ID')[COLS_A_IMPUTER].transform(
    lambda s: s.ffill().bfill()
)

missing_apres_imputation = df[COLS_A_IMPUTER].isnull().sum()

pd.DataFrame({
    'avant': missing_avant_imputation,
    'apres': missing_apres_imputation,
})


,avant,apres
Monthly_Inhand_Salary,15002,0
Occupation,7062,0
Credit_Mix,20195,0
Type_of_Loan,11408,11408
Payment_Behaviour,7600,0


## 6. Traitement des valeurs manquantes restantes de `Type_of_Loan`

Certains clients n'ont aucun mois renseigné pour `Type_of_Loan` (imputation par
client insuffisante). Dans ce cas, l'absence de prêt renseigné signifie
vraisemblablement que le client n'a pas de prêt : on remplace ces valeurs
manquantes restantes par `"No Loan"`.


In [17]:
n_avant = int(df['Type_of_Loan'].isnull().sum())

df['Type_of_Loan'] = df['Type_of_Loan'].fillna('No Loan')

n_apres = int(df['Type_of_Loan'].isnull().sum())
print(f"Valeurs manquantes dans Type_of_Loan avant : {n_avant}")
print(f"Valeurs manquantes dans Type_of_Loan après : {n_apres}")


Valeurs manquantes dans Type_of_Loan avant : 11408
Valeurs manquantes dans Type_of_Loan après : 0


## 7. Conversion de `Credit_History_Age` en nombre de mois

La colonne `Credit_History_Age` est au format texte `"X Years and Y Months"`.
On extrait les années et les mois pour calculer une nouvelle colonne numérique
`Credit_History_Age_Months` (= années x 12 + mois), plus facilement exploitable
pour la modélisation.


In [18]:
def convertir_en_mois(valeur):
    """Convertit '22 Years and 1 Months' en nombre total de mois (float)."""
    if pd.isna(valeur):
        return np.nan
    match = pd.Series([valeur]).str.extract(r'(\d+)\s+Years?\s+and\s+(\d+)\s+Months?')
    annees, mois = match.iloc[0]
    if pd.isna(annees) or pd.isna(mois):
        return np.nan
    return int(annees) * 12 + int(mois)

df['Credit_History_Age_Months'] = df['Credit_History_Age'].apply(convertir_en_mois)

print(f"Valeurs manquantes dans Credit_History_Age : {df['Credit_History_Age'].isnull().sum()}")
print(f"Valeurs manquantes dans Credit_History_Age_Months : {df['Credit_History_Age_Months'].isnull().sum()}")

df[['Credit_History_Age', 'Credit_History_Age_Months']].head(10)


Valeurs manquantes dans Credit_History_Age : 9030
Valeurs manquantes dans Credit_History_Age_Months : 9030


,Credit_History_Age,Credit_History_Age_Months
0,22 Years and 1 Months,265.0
1,NaN,NaN
2,22 Years and 3 Months,267.0
3,22 Years and 4 Months,268.0
4,22 Years and 5 Months,269.0
5,22 Years and 6 Months,270.0
6,22 Years and 7 Months,271.0
7,NaN,NaN
8,26 Years and 7 Months,319.0
9,26 Years and 8 Months,320.0


In [19]:
# Les valeurs manquantes restantes de Credit_History_Age_Months sont imputées
# par propagation avant/arrière groupée par Customer_ID, comme à l'étape 5.
n_avant = int(df['Credit_History_Age_Months'].isnull().sum())

df['Credit_History_Age_Months'] = df.groupby('Customer_ID')['Credit_History_Age_Months'].transform(
    lambda s: s.ffill().bfill()
)

n_apres = int(df['Credit_History_Age_Months'].isnull().sum())
print(f"Valeurs manquantes avant imputation par client : {n_avant}")
print(f"Valeurs manquantes après imputation par client : {n_apres}")


Valeurs manquantes avant imputation par client : 9030
Valeurs manquantes après imputation par client : 0


## 8. Imputation des colonnes restantes

Il reste des valeurs manquantes sur des colonnes non couvertes par les étapes
précédentes. On les impute à leur tour en s'appuyant sur le même `Customer_ID`
(un client suivi sur plusieurs mois a généralement un comportement stable) :

- **Médiane du même client** pour les colonnes numériques : `Num_of_Delayed_Payment`,
  `Changed_Credit_Limit`, `Amount_invested_monthly`, `Monthly_Balance`.
- **Mode (valeur la plus fréquente) du même client** pour la colonne catégorielle :
  `Payment_of_Min_Amount`.

S'il ne reste aucune valeur valide pour un client donné, on utilise en secours la
médiane (ou le mode) global de la colonne.

`Name` et `SSN` sont volontairement exclus : ce sont des identifiants sans valeur
prédictive, non destinés à être utilisés par le modèle.

**Avant l'imputation**, on neutralise aussi les valeurs numériques totalement
implausibles qui ont échappé aux étapes précédentes (par exemple des valeurs de
l'ordre de `1e+26` dans `Monthly_Balance`, largement hors de toute réalité
bancaire) : elles sont remplacées par `NaN` puis traitées comme les autres
valeurs manquantes par la médiane du même client.


In [20]:
COLS_MEDIANE_CLIENT = [
    'Num_of_Delayed_Payment',
    'Changed_Credit_Limit',
    'Amount_invested_monthly',
    'Monthly_Balance',
]
COL_MODE_CLIENT = 'Payment_of_Min_Amount'

# Neutralisation des valeurs numériques implausibles (ordres de grandeur
# absurdes, ex. 1e+26) qui ont échappé au nettoyage précédent
SEUIL_PLAUSIBILITE = 1_000_000
for col in COLS_MEDIANE_CLIENT:
    implausibles = df[col].abs() > SEUIL_PLAUSIBILITE
    if implausibles.any():
        print(f"{col} : {implausibles.sum()} valeur(s) implausible(s) détectée(s), "
              f"ex. {df.loc[implausibles, col].iloc[0]:.3e} -> mises à NaN")
        df.loc[implausibles, col] = np.nan

missing_avant_finales = df[COLS_MEDIANE_CLIENT + [COL_MODE_CLIENT]].isnull().sum()
missing_avant_finales


Monthly_Balance : 9 valeur(s) implausible(s) détectée(s), ex. -3.333e+26 -> mises à NaN


Num_of_Delayed_Payment      7002
Changed_Credit_Limit        2091
Amount_invested_monthly     8784
Monthly_Balance             1209
Payment_of_Min_Amount      12007
dtype: int64

In [21]:
def imputer_mediane_client(dataframe, colonne):
    """Comble les NaN par la médiane du même Customer_ID, avec repli sur la
    médiane globale si le client n'a aucune valeur valide."""
    serie = dataframe[colonne]
    mediane_client = serie.groupby(dataframe['Customer_ID']).transform('median')
    return serie.fillna(mediane_client).fillna(serie.median())


def imputer_mode_client(dataframe, colonne):
    """Comble les NaN par le mode (valeur la plus fréquente) du même
    Customer_ID, avec repli sur le mode global si le client n'a aucune valeur
    valide."""
    mode_global = dataframe[colonne].mode(dropna=True).iloc[0]

    def _mode_ou_nan(s):
        m = s.mode(dropna=True)
        return m.iloc[0] if not m.empty else np.nan

    mode_client = dataframe.groupby('Customer_ID')[colonne].transform(_mode_ou_nan)
    return dataframe[colonne].fillna(mode_client).fillna(mode_global)


for col in COLS_MEDIANE_CLIENT:
    df[col] = imputer_mediane_client(df, col)

df[COL_MODE_CLIENT] = imputer_mode_client(df, COL_MODE_CLIENT)


In [22]:
missing_apres_finales = df[COLS_MEDIANE_CLIENT + [COL_MODE_CLIENT]].isnull().sum()

pd.DataFrame({
    'avant': missing_avant_finales,
    'apres': missing_apres_finales,
})


,avant,apres
Num_of_Delayed_Payment,7002,0
Changed_Credit_Limit,2091,0
Amount_invested_monthly,8784,0
Monthly_Balance,1209,0
Payment_of_Min_Amount,12007,0


## 9. Vérification finale et résumé des corrections

On dresse un bilan complet : valeurs manquantes restantes, types de données
finaux, et comparaison globale avant/après nettoyage.


In [23]:
missing_final = df.isnull().sum()
missing_final = missing_final[missing_final > 0].sort_values(ascending=False)
print("Valeurs manquantes restantes par colonne :")
missing_final


Valeurs manquantes restantes par colonne :


Name                  9985
Credit_History_Age    9030
SSN                   5572
dtype: int64

In [24]:
df.dtypes


ID                               str
Customer_ID                      str
Month                            str
Name                             str
Age                          float64
SSN                              str
Occupation                       str
Annual_Income                float64
Monthly_Inhand_Salary        float64
Num_Bank_Accounts            float64
Num_Credit_Card              float64
Interest_Rate                float64
Num_of_Loan                  float64
Type_of_Loan                     str
Delay_from_due_date            int64
Num_of_Delayed_Payment       float64
Changed_Credit_Limit         float64
Num_Credit_Inquiries         float64
Credit_Mix                       str
Outstanding_Debt             float64
Credit_Utilization_Ratio     float64
Credit_History_Age               str
Payment_of_Min_Amount            str
Total_EMI_per_month          float64
Amount_invested_monthly      float64
Payment_Behaviour                str
Monthly_Balance              float64
C

In [25]:
missing_total_final = int(df.isnull().sum().sum())

resume = pd.DataFrame({
    'Étape': [
        "Chargement initial (NaN explicites)",
        "Après remplacement des placeholders",
        "Après nettoyage final (toutes étapes)",
    ],
    'Cellules manquantes': [
        missing_total_initial,
        missing_apres_placeholders,
        missing_total_final,
    ],
})
resume['% du dataset'] = (resume['Cellules manquantes'] / n_cells_total * 100).round(3)
resume


,Étape,Cellules manquantes,% du dataset
0,Chargement initial (NaN explicites),60071,2.145
1,Après remplacement des placeholders,118903,4.247
2,Après nettoyage final (toutes étapes),24587,0.878


In [26]:
print(f"Dimensions finales du dataset : {df.shape[0]} lignes x {df.shape[1]} colonnes")
df.describe(include='all').T


Dimensions finales du dataset : 100000 lignes x 29 colonnes


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID,100000,100000,0x1602,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer_ID,100000,12500,CUS_0xd40,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Month,100000,8,January,12500,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Name,90015,10139,Langep,44,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,100000.0,NaN,NaN,NaN,33.3146,10.773909,14.0,24.0,33.0,42.0,100.0
SSN,94428,12500,004-07-5839,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Occupation,100000,15,Lawyer,7096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Annual_Income,100000.0,NaN,NaN,NaN,50518.846732,38335.621871,7005.93,19344.27,37002.58,71689.68,277803.0
Monthly_Inhand_Salary,100000.0,NaN,NaN,NaN,4198.771619,3187.494355,303.645417,1626.761667,3096.378333,5961.745,15204.633333
Num_Bank_Accounts,100000.0,NaN,NaN,NaN,5.36884,2.593273,0.0,3.0,5.0,7.0,11.0


## 10. Export du jeu de données nettoyé

Le dataset nettoyé est exporté vers `data/train_cleaned.csv`, prêt pour les
étapes de feature engineering et de modélisation.


In [27]:
df.to_csv('../data/train_cleaned.csv', index=False)
print("Fichier exporté avec succès : data/train_cleaned.csv")
print(f"Dimensions exportées : {df.shape[0]} lignes x {df.shape[1]} colonnes")


Fichier exporté avec succès : data/train_cleaned.csv
Dimensions exportées : 100000 lignes x 29 colonnes
